## Phase 3, Attempt 1: class_weight='balanced'

The baseline treated every row equally during training, even though
legit outnumbers fraud ~578:1. `class_weight='balanced'` reweights
the loss so each class contributes equally in aggregate, fraud rows
get upweighted ~578x, legit rows get downweighted slightly.

Everything else (split, features, threshold) stays identical to the
baseline, so any change in the metrics is attributable to this one
lever. Expectation: recall up, precision down, the classic tradeoff,
made visible.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    average_precision_score,
)

df = pd.read_csv("data/raw/creditcard.csv")

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())  # sanity check: should both be ~0.0017, matching baseline

(227845, 30) (56962, 30)
0.001729245759178389 0.0017204452090867595


In [3]:
from sklearn.linear_model import LogisticRegression

model_balanced = LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42)
model_balanced.fit(X_train, y_train)

y_pred_balanced = model_balanced.predict(X_test)
y_scores_balanced = model_balanced.predict_proba(X_test)[:, 1]

cm_balanced = confusion_matrix(y_test, y_pred_balanced)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm_balanced)

precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)
auprc_balanced = average_precision_score(y_test, y_scores_balanced)

print(f"\nPrecision: {precision_balanced:.4f}  (baseline was 0.8312)")
print(f"Recall:    {recall_balanced:.4f}  (baseline was 0.6531)")
print(f"AUPRC:     {auprc_balanced:.4f}  (baseline was 0.7427)")

Confusion matrix [[TN, FP], [FN, TP]]:
[[55485  1379]
 [    8    90]]

Precision: 0.0613  (baseline was 0.8312)
Recall:    0.9184  (baseline was 0.6531)
AUPRC:     0.7159  (baseline was 0.7427)


## Phase 3, Attempt 1 result: class_weight='balanced'

| Metric    | Baseline | Balanced |
|-----------|----------|----------|
| Precision | 0.8312   | 0.0613   |
| Recall    | 0.6531   | 0.9184   |
| AUPRC     | 0.7427   | 0.7159   |

Confusion matrix: TP=90, FN=8, FP=1379, TN=55,485 (catches 90 of 98 frauds).

class_weight='balanced' massively improved recall (catches nearly all
fraud) but destroyed precision (1 in 40 legit transactions now falsely
flagged) likely unusable in practice as-is.

Notably, AUPRC barely changed (0.74 → 0.72). This means the model's
underlying ability to *rank* transactions by suspicion didn't really
improve  class_weight mainly shifted where the default 0.5 threshold
falls relative to that ranking, not the ranking itself. This suggests
threshold tuning, not just reweighting, is the next lever to pull.

## Phase 3, Attempt 2: threshold tuning

predict() applies a fixed 0.5 cutoff to the probability scores.
That cutoff is a default, not a requirement. We can apply any
threshold we want to y_scores_balanced ourselves, trading recall
against precision along the way.

Goal: find a threshold that keeps recall high (catching most fraud)
without precision collapsing to 0.06 the way it did at 0.5.

In [ ]:
import numpy as np

thresholds = np.arange(0.1, 0.95, 0.05)

print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10}")
for t in thresholds:
    y_pred_t = (y_scores_balanced >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    print(f"{t:>10.2f} {p:>10.4f} {r:>10.4f}")